Step 1 — Historical VaR & CVaR

In [27]:
import sqlite3
import pandas as pd
import numpy as np

# Connect database
conn = sqlite3.connect("../data/db/bluestock_mf.db")

# Load tables
funds = pd.read_sql(
    "SELECT * FROM dim_fund",
    conn
)

nav = pd.read_sql(
    "SELECT * FROM fact_nav",
    conn
)

perf = pd.read_sql(
    "SELECT * FROM fact_performance",
    conn
)

# Convert date column
nav["nav_date"] = pd.to_datetime(nav["nav_date"])

# Sort data
nav = nav.sort_values(
    ["amfi_code", "nav_date"]
)

print("Funds Shape :", funds.shape)
print("NAV Shape   :", nav.shape)
print("Perf Shape  :", perf.shape)

nav.head()

Funds Shape : (40, 13)
NAV Shape   : (46000, 5)
Perf Shape  : (40, 14)


,nav_id,amfi_code,nav_date,nav,daily_return_pct
0,1,100016,2022-01-03,520.4608,NaN
1,2,100016,2022-01-04,515.0971,-1.030568
2,3,100016,2022-01-05,521.7239,1.286515
3,4,100016,2022-01-06,515.7880,-1.137747
4,5,100016,2022-01-07,515.1639,-0.120999


In [28]:
var_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    returns = (
        group["daily_return_pct"]
        .dropna()
    )

    if len(returns) < 50:
        continue

    var95 = np.percentile(
        returns,
        5
    )

    cvar95 = returns[
        returns <= var95
    ].mean()

    var_results.append({

        "amfi_code": amfi_code,

        "var_95": var95,

        "cvar_95": cvar95

    })

var_cvar_df = pd.DataFrame(
    var_results
)

In [29]:
var_cvar_df = var_cvar_df.merge(

    funds[
        [
            "amfi_code",
            "scheme_name"
        ]
    ],

    on="amfi_code"

)

In [30]:
var_cvar_df.to_csv(
    "../reports/var_cvar_report.csv",
    index=False
)

Step 2 — Top Risk Funds

In [31]:
var_cvar_df.sort_values(
    "var_95"
).head(10)

,amfi_code,var_95,cvar_95,scheme_name
22,119599,-2.685944,-3.238412,SBI Small Cap Fund - Direct Plan - Growth
17,119095,-2.618842,-3.166729,Axis Small Cap Fund - Regular - Growth
4,101207,-2.602125,-3.245906,ABSL Small Cap Fund - Regular - Growth
11,118634,-2.543811,-3.230407,Nippon India Small Cap Fund - Regular - Growth
21,119598,-2.450705,-3.059526,SBI Small Cap Fund - Regular Plan - Growth
39,149324,-2.348307,-3.103625,DSP Small Cap Fund - Regular - Growth
7,102886,-1.922028,-2.325086,UTI Mid Cap Fund - Regular - Growth
2,100033,-1.903354,-2.345576,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
25,120505,-1.889179,-2.434207,ICICI Pru Midcap Fund - Regular - Growth
16,119094,-1.848028,-2.426006,Axis Midcap Fund - Regular - Growth


Step 2 — Rolling 90-Day Sharpe Ratio

In [32]:
scorecard = pd.read_csv(
    "../reports/fund_scorecard.csv"
)

scorecard.head()

,perf_id,amfi_code,as_of_date,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,...,scheme_name,category,expense_ratio_pct,return_rank,sharpe_rank,alpha_rank,expense_rank,drawdown_rank,fund_score,overall_rank
0,4,119599,2026-05-31,20.59,23.14,21.82,22.01,1.13,1.04,0.93,...,SBI Small Cap Fund - Direct Plan - Growth,Equity,0.72,0.975,0.5375,0.4625,0.8875,0.750,72.750,1
1,23,120843,2026-05-31,15.74,15.65,13.50,13.80,1.85,0.95,0.98,...,Kotak Flexicap Fund - Regular - Growth,Equity,1.45,0.750,0.7250,0.9375,0.4750,0.500,71.500,2
2,22,120842,2026-05-31,17.12,18.23,17.75,16.32,1.91,1.00,0.96,...,Kotak Emerging Equity Fund - Regular - Growth,Equity,1.56,0.850,0.6500,0.9750,0.2000,0.625,70.500,3
3,30,101207,2026-05-31,24.93,22.38,23.80,20.54,1.84,0.97,0.90,...,ABSL Small Cap Fund - Regular - Growth,Equity,1.53,0.950,0.4125,0.9000,0.3125,0.675,68.250,4
4,3,119598,2026-05-31,24.56,23.39,20.67,22.16,1.23,0.89,0.94,...,SBI Small Cap Fund - Regular Plan - Growth,Equity,1.43,1.000,0.5750,0.5250,0.5000,0.200,64.375,5


In [33]:
nav["amfi_code"] = nav["amfi_code"].astype(str)

scorecard["amfi_code"] = scorecard["amfi_code"].astype(str)

In [34]:
top5_codes = (
    scorecard
    .sort_values(
        "fund_score",
        ascending=False
    )
    .head(5)["amfi_code"]
    .tolist()
)

In [35]:
print(type(top5_codes[0]))

<class 'str'>


In [36]:
rolling_sharpe_df = pd.DataFrame()

for code in top5_codes:

    temp = nav[
        nav["amfi_code"] == code
    ].copy()

    temp = temp.sort_values(
        "nav_date"
    )

    temp["rolling_sharpe"] = (

        temp["daily_return_pct"]
        .rolling(90)
        .mean()

        /

        temp["daily_return_pct"]
        .rolling(90)
        .std()

    ) * np.sqrt(252)

    rolling_sharpe_df = pd.concat(
        [
            rolling_sharpe_df,
            temp
        ]
    )

In [37]:
rolling_sharpe_df = rolling_sharpe_df.merge(

    funds[
        [
            "amfi_code",
            "scheme_name"
        ]
    ],

    on="amfi_code",

    how="left"

)

In [38]:
import plotly.express as px

fig = px.line(

    rolling_sharpe_df,

    x="nav_date",

    y="rolling_sharpe",

    color="scheme_name",

    title="Rolling 90-Day Sharpe Ratio"

)

fig.show()

In [39]:
fig.write_image(
    "../reports/charts/rolling_sharpe_chart.png"
)

Task 3 — Investor Cohort Analysis

In [41]:
txn = pd.read_csv(
    "../data/raw/08_investor_transactions.csv"
)

In [43]:
txn.head()
txn.columns.tolist()

['investor_id',
 'transaction_date',
 'amfi_code',
 'transaction_type',
 'amount_inr',
 'state',
 'city',
 'city_tier',
 'age_group',
 'gender',
 'annual_income_lakh',
 'payment_mode',
 'kyc_status']

In [45]:
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"]
)

In [46]:
first_txn = (
    txn.groupby("investor_id")["transaction_date"]
    .min()
    .reset_index()
)

first_txn["cohort_year"] = (
    first_txn["transaction_date"]
    .dt.year
)

In [47]:
txn = txn.merge(

    first_txn[
        [
            "investor_id",
            "cohort_year"
        ]
    ],

    on="investor_id"

)

In [48]:
cohort_analysis = (

    txn.groupby("cohort_year")
    .agg(

        investors=("investor_id","nunique"),

        avg_investment=("amount_inr","mean"),

        total_invested=("amount_inr","sum")

    )

    .reset_index()

)

cohort_analysis

,cohort_year,investors,avg_investment,total_invested
0,2024,4803,107422.541832,3491125187
1,2025,197,109158.577061,30455243


In [49]:
fund_pref = (

    txn.groupby(
        [
            "cohort_year",
            "amfi_code"
        ]
    )

    .size()

    .reset_index(
        name="transactions"
    )

)

In [50]:
fund_pref = fund_pref.sort_values(
    ["cohort_year","transactions"],
    ascending=False
)

In [51]:
cohort_analysis.to_csv(
    "../reports/cohort_analysis.csv",
    index=False
)

Task 4 — SIP Continuity Analysis

In [52]:
sip_txn = txn[
    txn["transaction_type"] == "SIP"
].copy()

In [53]:
sip_txn = sip_txn.sort_values(

    [
        "investor_id",
        "transaction_date"
    ]

)

In [54]:
sip_txn["gap_days"] = (

    sip_txn.groupby(
        "investor_id"
    )["transaction_date"]

    .diff()

    .dt.days

)

In [55]:
sip_continuity = (

    sip_txn.groupby(
        "investor_id"
    )

    .agg(

        sip_count=(
            "transaction_date",
            "count"
        ),

        avg_gap_days=(
            "gap_days",
            "mean"
        )

    )

    .reset_index()

)

In [59]:
sip_continuity = sip_continuity[
    sip_continuity["sip_count"] >= 6
]

In [60]:
sip_continuity["risk_flag"] = np.where(

    sip_continuity[
        "avg_gap_days"
    ] > 35,

    "At Risk",

    "Healthy"

)

In [61]:
sip_continuity.to_csv(
    "../reports/sip_continuity.csv",
    index=False
)

Task 5 — Fund Recommendation Engine.

In [62]:
funds["risk_category"].value_counts()

risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

In [63]:
recommendation_df = funds.merge(

    perf[
        [
            "amfi_code",
            "sharpe_ratio",
            "return_3yr_pct",
            "alpha"
        ]
    ],

    on="amfi_code",

    how="inner"

)

In [64]:
def recommend_funds(risk_appetite):

    result = (

        recommendation_df[
            recommendation_df["risk_category"]
            .str.lower()
            ==
            risk_appetite.lower()
        ]

        .sort_values(
            "sharpe_ratio",
            ascending=False
        )

        .head(3)

        [
            [
                "scheme_name",
                "risk_category",
                "return_3yr_pct",
                "sharpe_ratio",
                "alpha"
            ]
        ]

    )

    return result

In [65]:
recommend_funds("Low")

,scheme_name,risk_category,return_3yr_pct,sharpe_ratio,alpha
14,ICICI Pru Liquid Fund - Regular - Growth,Low,7.68,7.68,1.85
23,Kotak Liquid Fund - Regular - Growth,Low,6.18,6.18,1.52
30,ABSL Liquid Fund - Regular - Growth,Low,5.14,5.14,1.18


In [66]:
recommend_funds("Moderate")

,scheme_name,risk_category,return_3yr_pct,sharpe_ratio,alpha
5,HDFC Top 100 Fund - Regular Plan - Growth,Moderate,14.84,1.06,0.78
34,Mirae Asset Large Cap Fund - Regular - Growth,Moderate,14.81,1.06,1.62
11,ICICI Pru Bluechip Fund - Direct - Growth,Moderate,14.41,1.03,0.88


In [67]:
recommend_funds("Very High")

,scheme_name,risk_category,return_3yr_pct,sharpe_ratio,alpha
2,SBI Small Cap Fund - Regular Plan - Growth,Very High,23.39,0.94,1.23
3,SBI Small Cap Fund - Direct Plan - Growth,Very High,23.14,0.93,1.13
29,ABSL Small Cap Fund - Regular - Growth,Very High,22.38,0.90,1.84


Task 6 — Sector Concentration Risk (HHI)

In [68]:
portfolio = pd.read_sql(
    "SELECT * FROM fact_portfolio",
    conn
)

portfolio.head()

,holding_id,amfi_code,as_of_date,stock_symbol,stock_name,sector,weight_pct
0,1,119551,None,POWERGRID,Power Grid Corporation,Utilities,13.85
1,2,119551,None,HDFCBANK,HDFC Bank Ltd,Banking,11.19
2,3,119551,None,GRASIM,Grasim Industries Ltd,Diversified,9.90
3,4,119551,None,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76
4,5,119551,None,ASIANPAINT,Asian Paints Ltd,Paints,10.25


In [69]:
portfolio.columns.tolist()

['holding_id',
 'amfi_code',
 'as_of_date',
 'stock_symbol',
 'stock_name',
 'sector',
 'weight_pct']

In [70]:
portfolio["weight_decimal"] = (
    portfolio["weight_pct"] / 100
)

In [71]:
sector_hhi = (

    portfolio

    .groupby("amfi_code")

    .apply(

        lambda x: np.sum(
            x["weight_decimal"] ** 2
        )

    )

    .reset_index(
        name="hhi"
    )

)

C:\Users\SUJAL SARNOBAT\AppData\Local\Temp\ipykernel_28744\1311563409.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [72]:
sector_hhi = sector_hhi.merge(

    funds[
        [
            "amfi_code",
            "scheme_name",
            "category"
        ]
    ],

    on="amfi_code",

    how="left"

)

In [73]:
sector_hhi["concentration_level"] = np.select(

    [

        sector_hhi["hhi"] < 0.15,

        sector_hhi["hhi"] < 0.25

    ],

    [

        "Diversified",

        "Moderate"

    ],

    default="Highly Concentrated"

)

In [74]:
sector_hhi.sort_values(
    "hhi",
    ascending=False
).head(10)

,amfi_code,hhi,scheme_name,category,concentration_level
11,119092,0.206448,Axis Bluechip Fund - Regular - Growth,Equity,Moderate
3,101207,0.200700,ABSL Small Cap Fund - Regular - Growth,Equity,Moderate
18,119599,0.174751,SBI Small Cap Fund - Direct Plan - Growth,Equity,Moderate
4,102885,0.174709,UTI Nifty 50 Index Fund - Regular - Growth,Equity,Moderate
7,118632,0.168298,Nippon India Large Cap Fund - Regular - Growth,Equity,Moderate
29,148568,0.167930,Mirae Asset Emerging Bluechip Fund - Regular -...,Equity,Moderate
21,120505,0.157570,ICICI Pru Midcap Fund - Regular - Growth,Equity,Moderate
22,120506,0.153794,ICICI Pru Value Discovery Fund - Regular - Growth,Equity,Moderate
27,125498,0.152414,HDFC Mid-Cap Opportunities Fund - Direct - Growth,Equity,Moderate
23,120841,0.149680,Kotak Bluechip Fund - Regular - Growth,Equity,Diversified


In [76]:
sector_hhi.sort_values(
    "hhi"
).head(10)

,amfi_code,hhi,scheme_name,category,concentration_level
17,119598,0.107349,SBI Small Cap Fund - Regular Plan - Growth,Equity,Diversified
16,119552,0.108011,SBI Bluechip Fund - Direct Plan - Growth,Equity,Diversified
9,118634,0.108358,Nippon India Small Cap Fund - Regular - Growth,Equity,Diversified
20,120504,0.108674,ICICI Pru Bluechip Fund - Direct - Growth,Equity,Diversified
14,119095,0.109605,Axis Small Cap Fund - Regular - Growth,Equity,Diversified
5,102886,0.114693,UTI Mid Cap Fund - Regular - Growth,Equity,Diversified
33,149324,0.118677,DSP Small Cap Fund - Regular - Growth,Equity,Diversified
15,119551,0.118716,SBI Bluechip Fund - Regular Plan - Growth,Equity,Diversified
8,118633,0.121461,Nippon India Large Cap Fund - Direct - Growth,Equity,Diversified
24,120842,0.127439,Kotak Emerging Equity Fund - Regular - Growth,Equity,Diversified


In [77]:
sector_hhi.to_csv(
    "../reports/sector_hhi.csv",
    index=False
)

In [78]:
import plotly.express as px

top_hhi = (
    sector_hhi
    .sort_values(
        "hhi",
        ascending=False
    )
    .head(10)
)

In [79]:
fig = px.bar(

    top_hhi,

    x="hhi",

    y="scheme_name",

    color="concentration_level",

    orientation="h",

    title="Top 10 Most Concentrated Funds (HHI)"

)

fig.show()

In [80]:
fig.write_image(
    "../reports/charts/sector_hhi_chart.png"
)